<a href="https://colab.research.google.com/github/snehiljaiswalsneh/Sales-Data-Filter-Report-Engine/blob/main/Sales_Data_Filter_Report_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Sales Data Filter & Report Engine
# Simulates one year of sales for 10 stores and builds filters and reports.

import numpy as np

NUM_STORES = 10
NUM_MONTHS = 12


# ---------------------------------------------------------------
# Step 1: Create the sales dataset
# ---------------------------------------------------------------
def create_dataset():
    # Month numbers 1 to 12 using np.arange()
    month_numbers = np.arange(1, 13)
    month_names = ["January", "February", "March", "April", "May", "June",
                   "July", "August", "September", "October", "November", "December"]

    # Fixed seed so the output is the same every time the program runs
    np.random.seed(42)

    # 120 random sales values (10 stores x 12 months), then reshape to 10 rows, 12 columns
    random_sales = np.random.randint(30000, 90000, size=NUM_STORES * NUM_MONTHS)
    sales = random_sales.reshape(NUM_STORES, NUM_MONTHS)

    # Monthly targets rising steadily from 50000 to 70000 (one target per month, per store)
    targets = np.linspace(50000, 70000, NUM_MONTHS)

    return month_numbers, month_names, sales, targets


# ---------------------------------------------------------------
# Step 2: Display the sales data
# ---------------------------------------------------------------
def display_sales_data(month_numbers, month_names, sales, targets):
    print("=" * 60)
    print("STEP 2: SALES DATA")
    print("=" * 60)

    print("\nMonths:")
    for i in range(NUM_MONTHS):
        print("  Month", month_numbers[i], "-", month_names[i])

    print("\nSales of each store (rows = stores, columns = months 1 to 12):")
    for store in range(NUM_STORES):
        print("Store", store + 1, ":", sales[store])

    print("\nMonthly Targets (per store):")
    for i in range(NUM_MONTHS):
        print(f"  {month_names[i]:<10} : {targets[i]:10.2f}")


# ---------------------------------------------------------------
# Step 3: 2D slicing
# ---------------------------------------------------------------
def perform_slicing(sales):
    print("\n" + "=" * 60)
    print("STEP 3: 2D SLICING")
    print("=" * 60)

    q1 = sales[:, 0:3]     # January to March
    q2 = sales[:, 3:6]     # April to June
    q3 = sales[:, 6:9]     # July to September
    q4 = sales[:, 9:12]    # October to December

    print("\nQ1 (Jan-Mar) sales, all stores:")
    print(q1)
    print("\nQ2 (Apr-Jun) sales, all stores:")
    print(q2)
    print("\nQ3 (Jul-Sep) sales, all stores:")
    print(q3)
    print("\nQ4 (Oct-Dec) sales, all stores:")
    print(q4)

    # Complete record of one store (Store 3 is row index 2)
    print("\nComplete sales record of Store 3:")
    print(sales[2, :])


# ---------------------------------------------------------------
# Step 4: Boolean masking
# ---------------------------------------------------------------
def perform_boolean_masking(sales, targets, month_names):
    print("\n" + "=" * 60)
    print("STEP 4: BOOLEAN MASKING")
    print("=" * 60)

    # Query 1: sales above the monthly target
    # targets has 12 values, so each column is compared with its own month's target
    above_target = sales > targets
    print("\nQuery 1: Store-month records where sales exceed the monthly target")
    print("Total records above target:", above_target.sum(), "out of", NUM_STORES * NUM_MONTHS)
    for store in range(NUM_STORES):
        for month in range(NUM_MONTHS):
            if above_target[store, month]:
                print(f"  Store {store + 1}, {month_names[month]}: "
                      f"{sales[store, month]} (target {targets[month]:.0f})")

    # Query 2: stores whose average sales are below the overall average
    store_average = sales.mean(axis=1)
    overall_average = sales.mean()
    below_average = store_average < overall_average
    print("\nQuery 2: Stores with average sales below the overall average")
    print(f"Overall average sales: {overall_average:.2f}")
    for store in range(NUM_STORES):
        if below_average[store]:
            print(f"  Store {store + 1}: average {store_average[store]:.2f}")

    # Query 3: AND condition - sales between 50000 and 70000
    and_mask = (sales >= 50000) & (sales <= 70000)
    print("\nQuery 3 (AND): Sales between 50000 and 70000")
    print("Number of records:", and_mask.sum())
    print("First 10 matching values:", sales[and_mask][:10])

    # Query 4: OR condition - very low (< 35000) or very high (> 85000) sales
    or_mask = (sales < 35000) | (sales > 85000)
    print("\nQuery 4 (OR): Sales below 35000 OR above 85000")
    print("Number of records:", or_mask.sum())
    print("Matching values:", sales[or_mask])

    # Query 5: NOT condition - records that did NOT exceed the target
    not_mask = ~above_target
    print("\nQuery 5 (NOT): Records that did NOT exceed the monthly target")
    print("Number of records:", not_mask.sum())
    print("First 10 matching values:", sales[not_mask][:10])


# ---------------------------------------------------------------
# Step 5: Fancy indexing
# ---------------------------------------------------------------
def perform_fancy_indexing(sales, month_names):
    print("\n" + "=" * 60)
    print("STEP 5: FANCY INDEXING")
    print("=" * 60)

    # These are zero-based indices: 1 = Store 2, 4 = Store 5, 7 = Store 8
    store_indices = np.array([1, 4, 7])
    selected_stores = sales[store_indices]

    print("\nSales of Store 2, Store 5 and Store 8:")
    for i in range(3):
        print(f"  Store {store_indices[i] + 1}: {selected_stores[i]}")

    # Total sales of each month across all stores
    monthly_totals = sales.sum(axis=0)

    # Find the 3 best months (largest totals) with a simple loop
    best_list = []
    for rank in range(3):
        best_month = -1
        best_value = -1
        for month in range(NUM_MONTHS):
            if month not in best_list and monthly_totals[month] > best_value:
                best_value = monthly_totals[month]
                best_month = month
        best_list.append(best_month)

    # Use the index array to select the best months and their totals
    best_indices = np.array(best_list)
    best_names = np.array(month_names)[best_indices]
    best_totals = monthly_totals[best_indices]

    print("\nThree best-performing months (by total sales of all stores):")
    for i in range(3):
        print(f"  {i + 1}. {best_names[i]} - total sales {best_totals[i]}")


# ---------------------------------------------------------------
# Step 6: Reports
# ---------------------------------------------------------------
def generate_reports(sales, targets, month_names):
    print("\n" + "=" * 60)
    print("STEP 6: REPORTS")
    print("=" * 60)

    monthly_totals = sales.sum(axis=0)
    total_targets = targets * NUM_STORES   # target for all 10 stores together

    # Report 1: Monthly report
    print("\nREPORT 1: MONTHLY REPORT")
    print(f"{'Month':<10} {'Total Sales':>12} {'Total Target':>13} {'Achievement':>12}")
    for month in range(NUM_MONTHS):
        achievement = monthly_totals[month] / total_targets[month] * 100
        print(f"{month_names[month]:<10} {monthly_totals[month]:>12} "
              f"{total_targets[month]:>13.0f} {achievement:>11.1f}%")

    # Report 2: Store-wise report
    print("\nREPORT 2: STORE-WISE REPORT")
    print(f"{'Store':<8} {'Annual Sales':>13} {'Monthly Average':>16}")
    for store in range(NUM_STORES):
        annual = sales[store, :].sum()
        average = sales[store, :].mean()
        print(f"Store {store + 1:<2} {annual:>13} {average:>16.2f}")

    # Report 3: Quarterly report
    print("\nREPORT 3: QUARTERLY REPORT")
    print("Q1 (Jan-Mar):", sales[:, 0:3].sum())
    print("Q2 (Apr-Jun):", sales[:, 3:6].sum())
    print("Q3 (Jul-Sep):", sales[:, 6:9].sum())
    print("Q4 (Oct-Dec):", sales[:, 9:12].sum())

    # Report 4: Growth target report
    print("\nREPORT 4: GROWTH TARGET REPORT")
    print(f"{'Month':<10} {'Avg Sales/Store':>16} {'Target':>10} {'Difference':>12}  Status")
    for month in range(NUM_MONTHS):
        average_sales = sales[:, month].mean()
        difference = average_sales - targets[month]
        if difference >= 0:
            status = "Target Met"
        else:
            status = "Target Missed"
        print(f"{month_names[month]:<10} {average_sales:>16.2f} {targets[month]:>10.2f} "
              f"{difference:>12.2f}  {status}")


# ---------------------------------------------------------------
# Main program
# ---------------------------------------------------------------
def main():
    month_numbers, month_names, sales, targets = create_dataset()
    display_sales_data(month_numbers, month_names, sales, targets)
    perform_slicing(sales)
    perform_boolean_masking(sales, targets, month_names)
    perform_fancy_indexing(sales, month_names)
    generate_reports(sales, targets, month_names)


main()


STEP 2: SALES DATA

Months:
  Month 1 - January
  Month 2 - February
  Month 3 - March
  Month 4 - April
  Month 5 - May
  Month 6 - June
  Month 7 - July
  Month 8 - August
  Month 9 - September
  Month 10 - October
  Month 11 - November
  Month 12 - December

Sales of each store (rows = stores, columns = months 1 to 12):
Store 1 : [86422 45795 30860 68158 84343 74732 41284 84886 36265 46850 67194 51962]
Store 2 : [77191 74131 46023 71090 31685 30769 89735 86101 32433 35311 67819 69188]
Store 3 : [47568 83707 49769 58693 36396 57480 71434 55658 48942 87001 48431 32747]
Store 4 : [88319 89150 30189 49118 65773 83810 31899 86886 31267 61551 80680 41394]
Store 5 : [33556 33890 71606 60740 44502 51777 40627 38792 73323 73021 38433 73001]
Store 6 : [41016 53897 32612 86193 88707 85387 84638 53483 88822 78555 89900 47159]
Store 7 : [74974 70994 44541 65920 56531 31585 33943 53939 49457 31021 41653 73573]
Store 8 : [43417 82995 70757 39692 75758 36873 35675 30161 67065 56557 63763 62606]
Sto